In [1]:
import re
import os
import time
import json
import calendar
from dateutil import parser
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor

import requests
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from scipy.interpolate import RBFInterpolator
from scipy.interpolate import PchipInterpolator

from api_client import TradingDeskAPI
from options import OptionSurface, Deribit, OKX, Bybit
from scanner import MarketScanner

import uuid

In [2]:
load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
BASE_URL = os.getenv("BASE_URL")
USER_EMAIL = os.getenv("USER_EMAIL")
USER_PASSWORD = os.getenv("USER_PASSWORD")
print(BASE_URL)

target_expiry_str = "25DEC26"
target_expiry = datetime.strptime(target_expiry_str, "%d%b%y").strftime("%Y%m%d")

currencies = ["BTC", "ETH"]

s = OptionSurface()

deribit = Deribit(currencies=currencies, target_expiry=target_expiry)
okx = OKX(currencies=currencies, target_expiry=target_expiry)
bybit = Bybit(currencies=currencies, target_expiry=target_expiry)

s.initialize(currencies=currencies, exchanges=[deribit, okx, bybit])

https://alphasignal-dev.moretoncp.com
spot: 64117.0 volume24h: 7.4046
spot: 1879.9 volume24h: 36.6465
spot: 64303.2 volume24h: 3525.23033863
spot: 1885.54 volume24h: 67864.811808
spot: 64289.2 volume24h: 5511.27929
spot: 1885.13 volume24h: 59728.63367


In [3]:
api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD)

markets = api.get_markets(limit=10000, liquidity_num_min=10000, volume_num_min=5000)

all_markets_df = pd.DataFrame(markets)

all_markets_df.head()

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,oneYearPriceChange,gameStartTime,secondsDelay,sportsMarketType,line,eventStartTime,umaResolutionStatus,gameId,marketMetadata,groupItemRange
0,677396,Iran Nuke before 2027?,0x8bdeac60c92d3bc494792fd334ca181b0cf70355f23d...,iran-nuke-before-2027,,2026-12-31T00:00:00Z,99987.87064,2025-11-13T23:10:38.346766Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,628947,Will Chad Bianco win the California Governor E...,0x1a5b898bbfa1e697c19afeef356d9c1a6ecb95d2493c...,will-chad-bianco-win-the-california-governor-e...,,2026-11-03T00:00:00Z,99767.13628,2025-10-09T23:29:51.802274Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1343228,"Will Bitcoin dip to $5,000 by December 31, 2026?",0xe681a6326237f3b17ce8622728b6cd104281dfe4d2b3...,will-bitcoin-dip-to-5000-by-december-31-2026-j...,,2027-01-01T05:00:00Z,99732.30818,2026-02-05T22:18:33.034Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1731346,Iran agrees to surrender enriched uranium stoc...,0xe846dd72f8a654ef137a3e23a88226400b42cc0ca817...,iran-agrees-to-surrender-enriched-uranium-stoc...,,2026-12-31T00:00:00Z,99425.5347,2026-03-27T00:15:17.118Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2589812,Will there be no change in Fed interest rates ...,0xdf9bf27ee5757c55b44b8b9826ddc9ec3a8809aa3278...,will-there-be-no-change-in-fed-interest-rates-...,,2026-10-28T23:59:00Z,99411.0236,2026-06-18T00:02:51.727205Z,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
BTC_KEYWORDS = [
    "bitcoin",
    "btc",
    "xbt"
]

ETH_KEYWORDS = [
    "ethereum",
    " eth "
]

scanner = MarketScanner(api=api, BASE_URL=BASE_URL, USER_EMAIL=USER_EMAIL, USER_PASSWORD=USER_PASSWORD)
markets_df, opportunities_df = scanner.scan_market(markets_df=all_markets_df, s=s, KEYWORDS=BTC_KEYWORDS)

opportunities_df

iv 0.635223284866965
p_touch_above: 1.0 p_touch_below: 0.0

BTC
buy_yes_ev: -0.02030473
sell_yes_ev: 0.01676268
buy_no_ev: 0.01676268000000003
sell_no_ev: -0.020304729999999993
buy_yes_kelly: -0.010362778417823739
sell_yes_kelly: 0.5
buy_no_kelly: 0.5
sell_no_kelly: -0.010362778417823735
iv 0.6615337487456303
p_touch_above: 1.0 p_touch_below: 0.00779

BTC
buy_yes_ev: 0.0013725199999999995
sell_yes_ev: -0.0031382500000000004
buy_no_ev: -0.0031382500000000455
sell_no_ev: 0.0013725200000000246
buy_yes_kelly: 0.0006906925053391637
sell_yes_kelly: -0.33731928843983455
buy_no_kelly: -0.33731928843984266
sell_no_kelly: 0.0006906925053391763
iv 0.6200195535086371
p_touch_above: 1.0 p_touch_below: 0.01436

BTC
buy_yes_ev: 0.0036669999999999984
sell_yes_ev: -0.0059843299999999995
buy_no_ev: -0.005984329999999942
sell_no_ev: 0.003666999999999936
buy_yes_kelly: 0.0018533175242872021
sell_yes_kelly: -0.35724485324756106
buy_no_kelly: -0.3572448532475552
sell_no_kelly: 0.001853317524287171
iv 0.4200

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,buy_yes_fee,sell_yes_fee,buy_no_fee,sell_no_fee,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly,best_ev,best_action
98,2467208,"Will Bitcoin reach $80,000 by December 31, 2026?",0xc564e47b7a853f3e52ea7b8e28d69ed99fcb28492936...,will-bitcoin-reach-80000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,85666.8276,2026-06-08T04:58:16.592002Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-2.781800e-02,-0.013367,-0.013367,-2.781800e-02,-2.158804e-02,-0.021250,-0.021250,-2.158804e-02,0.015708,buy_yes_ev
55,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,91169.9551,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-4.942300e-02,0.010310,0.010310,-4.942300e-02,-8.966860e-02,0.007522,0.007522,-8.966860e-02,0.014700,sell_yes_ev
48,701496,"Will Bitcoin reach $100,000 by December 31, 2026?",0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49db...,will-bitcoin-reach-100000-by-december-31-2026-...,,2027-01-01T05:00:00Z,92135.9579,2025-11-24T19:07:17.691Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-2.775300e-02,0.006868,0.006868,-2.775300e-02,-1.534558e-02,0.045880,0.045880,-1.534558e-02,0.005733,buy_yes_ev
56,1339769,"Will Bitcoin dip to $30,000 by December 31, 2026?",0x903a1387e6414be5529efc8f7757e30371e6698de9d6...,will-bitcoin-dip-to-30000-by-december-31-2026-...,,2027-01-01T05:00:00Z,91119.1525,2026-02-05T14:48:05.459Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,2.229200e-02,-0.039565,-0.039565,2.229200e-02,1.190746e-02,-0.423835,-0.423835,1.190746e-02,0.003948,sell_no_ev
2,1343228,"Will Bitcoin dip to $5,000 by December 31, 2026?",0xe681a6326237f3b17ce8622728b6cd104281dfe4d2b3...,will-bitcoin-dip-to-5000-by-december-31-2026-j...,,2027-01-01T05:00:00Z,99732.30818,2026-02-05T22:18:33.034Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-2.030473e-02,0.016763,0.016763,-2.030473e-02,-1.036278e-02,0.500000,0.500000,-1.036278e-02,0.001305,sell_no_ev
20,3257366,"Will Bitcoin dip to $45,000 in August?",0x462dc3a25f4bd83645256c18ef2ad472b93135a29b4f...,will-bitcoin-dip-to-45k-in-august-2026,NaN,2026-09-01T04:00:00Z,97250.34567,2026-08-01T05:07:40Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,3.667000e-03,-0.005984,-0.005984,3.667000e-03,1.853318e-03,-0.357245,-0.357245,1.853318e-03,0.000693,sell_no_ev
9,3257368,"Will Bitcoin dip to $42,500 in August?",0x98e5997750cfc27c47ad3b4f4278a01935bed1edfcec...,will-bitcoin-dip-to-42pt5k-in-august-2026,NaN,2026-09-01T04:00:00Z,98823.34228,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,1.372520e-03,-0.003138,-0.003138,1.372520e-03,6.906925e-04,-0.337319,-0.337319,6.906925e-04,0.000417,sell_no_ev
95,3257332,"Will Bitcoin reach $100,000 in August?",0x39ea1503427df7f1439e7dd6567aad5ed72284ed49c5...,will-bitcoin-reach-100k-in-august-2026,NaN,2026-09-01T04:00:00Z,86182.91595,2026-08-01T05:07:39Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,2.800000e-07,-0.001210,-0.001210,2.800000e-07,1.403002e-07,-0.650451,-0.650451,1.403002e-07,0.000140,sell_no_ev


In [5]:
# get all dfs
DATA_DIR = f"data"

orders_df = pd.read_parquet(f"{DATA_DIR}/orders.parquet")
fills_df = pd.read_parquet(f"{DATA_DIR}/fills.parquet")
positions_df = pd.read_parquet(f"{DATA_DIR}/positions.parquet")
realized_pnl_df = pd.read_parquet(f"{DATA_DIR}/realized_pnl.parquet")
equity_df = pd.read_parquet(f"{DATA_DIR}/equity.parquet")

# fills_df = fills_df.iloc[0:0]

In [ ]:
DATA_DIR = f"data"
os.makedirs(DATA_DIR, exist_ok=True)
filename = f"{DATA_DIR}/orders.parquet"
orders_df.to_parquet(filename, engine="fastparquet", index=False)

In [ ]:
# orders_df= orders_df.iloc[0:0] 

In [10]:
fills_df

,order_id,condition_id,token_id,outcome,side,price,shares,fee,timestamp


In [ ]:
# Inventory Management Step

def format_signal(best_action, row, pos, current_bid, hold_ev, exit_ev, mtm_pnl, EXIT_EV_THRESHOLD):

    print("============================================================")
    print(f'{"EXIT" if best_action == "exit" else "HOLD"} SIGNAL')
    print("============================================================")

    print(f"Market:\n{row["question"]}\n\n")
    print(f"Position:\n{pos["outcome"]}\n\n")
    print(f"Shares:\n{pos["shares"]}\n\n")
    print(f"Current Bid:\n{current_bid}\n\n")
    print(f"Exit EV:\n{exit_ev}\n\n")
    print(f"Hold EV:\n{hold_ev}\n\n")
    print(f"MTM Pnl:\n{mtm_pnl}\n\n")
    print(f"Exit_EV_threshold:\n{EXIT_EV_THRESHOLD}")


EXIT_EV_THRESHOLD = -0.02
 
positions = api.get_positions()
# positions =
# [
#     {
#         "condition_id": "0x7b9072e6...",
#         "token_id": "123456789...",
#         "outcome": "Yes",
#         "shares": 25.0
#     },
#     {
#         "condition_id": "0xdaa4866b...",
#         "token_id": "987654321...",
#         "outcome": "No",
#         "shares": 10.0
#     }
# ]

for pos in positions:

    condition_id = pos["condition_id"]
    row = markets_df.loc[markets_df["conditionId"] == condition_id]

    if pos["outcome"] == "Yes": # no shorting in polymarket
        current_bid = row["yes_bid"]
        hold_ev = row["buy_yes_ev"]
        exit_ev = row["sell_yes_ev"]

    elif pos["outcome"] == "No":
        current_bid = row["no_bid"]
        hold_ev = row["buy_no_ev"]
        exit_ev = row["sell_no_ev"]

    if hold_ev < EXIT_EV_THRESHOLD:
        best_action = "exit"

    else:
        best_action = "hold"

    size = pos["shares"]

    positions_df = reconstruct_positions_fifo(fills_df)

    # entry_price = pos.entry_price
    # entry_price = get_trades()
    entry_price = positions_df.iloc[positions_df["condition_id"] == condition_id]["avg_entry_price"]
    mtm_pnl = (current_bid - entry_price) * size

    format_signal(best_action, row, pos, current_bid, hold_ev, exit_ev, mtm_pnl, EXIT_EV_THRESHOLD)

    if best_action == "exit":
        confirm_order = (input("Place this order? Type YES to confirm: ") == "YES")

        if confirm_order == True:

            price_tick = api.get_tick_size(row["token_id"])
            price = api.round_to_tick(current_bid, price_tick)
            size = api.normalize_size(size)
            
            try:
                order = api.place_limit_order(
                    token_id=row["token_id"],
                    side="SELL",
                    price=price,
                    size=size,
                    order_type="GTC",
                )
        
                print("\nORDER SUBMITTED\n\n")
                print(order)
        
                order_row = {
                    "order_id": order["clob_order_id"],
                    "condition_id": condition_id,
                    "token_id": row["token_id"],
                    "outcome": pos["outcome"],
                    "side": "SELL",
                    "price": price,
                    "requested_size": size,
                    "order_type": "GTC",
                    "status": "OPEN",
                    "created_at": datetime.now(timezone.utc),
                    "cancelled_at": None,
                }

                orders_df = pd.concat([orders_df, pd.DataFrame([order_row])], ignore_index=True)
        
                print("\nORDER RECORDED\n\n")
    
            except Exception as e:
                print("\nORDER ERROR\n\n")
                print(e)
                break

        else:
            print("\nSKIPPING ORDER\n\n")

In [ ]:
tick = api.get_tick_size(1)

In [8]:
opportunities_df

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,buy_yes_fee,sell_yes_fee,buy_no_fee,sell_no_fee,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly,best_ev,best_action
98,2467208,"Will Bitcoin reach $80,000 by December 31, 2026?",0xc564e47b7a853f3e52ea7b8e28d69ed99fcb28492936...,will-bitcoin-reach-80000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,85666.8276,2026-06-08T04:58:16.592002Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-2.781800e-02,-0.013367,-0.013367,-2.781800e-02,-2.158804e-02,-0.021250,-0.021250,-2.158804e-02,0.015708,buy_yes_ev
55,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,91169.9551,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-4.942300e-02,0.010310,0.010310,-4.942300e-02,-8.966860e-02,0.007522,0.007522,-8.966860e-02,0.014700,sell_yes_ev
48,701496,"Will Bitcoin reach $100,000 by December 31, 2026?",0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49db...,will-bitcoin-reach-100000-by-december-31-2026-...,,2027-01-01T05:00:00Z,92135.9579,2025-11-24T19:07:17.691Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-2.775300e-02,0.006868,0.006868,-2.775300e-02,-1.534558e-02,0.045880,0.045880,-1.534558e-02,0.005733,buy_yes_ev
56,1339769,"Will Bitcoin dip to $30,000 by December 31, 2026?",0x903a1387e6414be5529efc8f7757e30371e6698de9d6...,will-bitcoin-dip-to-30000-by-december-31-2026-...,,2027-01-01T05:00:00Z,91119.1525,2026-02-05T14:48:05.459Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,2.229200e-02,-0.039565,-0.039565,2.229200e-02,1.190746e-02,-0.423835,-0.423835,1.190746e-02,0.003948,sell_no_ev
2,1343228,"Will Bitcoin dip to $5,000 by December 31, 2026?",0xe681a6326237f3b17ce8622728b6cd104281dfe4d2b3...,will-bitcoin-dip-to-5000-by-december-31-2026-j...,,2027-01-01T05:00:00Z,99732.30818,2026-02-05T22:18:33.034Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-2.030473e-02,0.016763,0.016763,-2.030473e-02,-1.036278e-02,0.500000,0.500000,-1.036278e-02,0.001305,sell_no_ev
20,3257366,"Will Bitcoin dip to $45,000 in August?",0x462dc3a25f4bd83645256c18ef2ad472b93135a29b4f...,will-bitcoin-dip-to-45k-in-august-2026,NaN,2026-09-01T04:00:00Z,97250.34567,2026-08-01T05:07:40Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,3.667000e-03,-0.005984,-0.005984,3.667000e-03,1.853318e-03,-0.357245,-0.357245,1.853318e-03,0.000693,sell_no_ev
9,3257368,"Will Bitcoin dip to $42,500 in August?",0x98e5997750cfc27c47ad3b4f4278a01935bed1edfcec...,will-bitcoin-dip-to-42pt5k-in-august-2026,NaN,2026-09-01T04:00:00Z,98823.34228,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,1.372520e-03,-0.003138,-0.003138,1.372520e-03,6.906925e-04,-0.337319,-0.337319,6.906925e-04,0.000417,sell_no_ev
95,3257332,"Will Bitcoin reach $100,000 in August?",0x39ea1503427df7f1439e7dd6567aad5ed72284ed49c5...,will-bitcoin-reach-100k-in-august-2026,NaN,2026-09-01T04:00:00Z,86182.91595,2026-08-01T05:07:39Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,2.800000e-07,-0.001210,-0.001210,2.800000e-07,1.403002e-07,-0.650451,-0.650451,1.403002e-07,0.000140,sell_no_ev


In [11]:
# New Opportunities Step

def format_buy_signal(trade, outcome, best_action, normalized_size, current_ask,
                      unrealized_pnl, bankroll, MIN_EV_THRESHOLD, MAX_POSITION):

    print("============================================================")
    print(f"BUY SIGNAL")
    print("============================================================")

    print(f"Market:\n{trade["question"]}\n\n")
    print(f"Outcome:\n{outcome}\n\n")
    print(f"Best Action:\n{best_action}\n\n")
    print(f"Recommended Size:\n{normalized_size}\n\n")
    print(f"Current Ask:\n{current_ask}\n\n")
    print(f"Buy Yes EV:\n{trade["buy_yes_ev"]}\n\n")
    print(f"Sell No EV:\n{trade["sell_no_ev"]}\n\n")
    print(f"Buy No EV:\n{trade["buy_no_ev"]}\n\n")
    print(f"Sell Yes EV:\n{trade["sell_yes_ev"]}\n\n")
    print(f"Unrealized Pnl:\n{unrealized_pnl}\n\n")
    print(f"Bankroll:\n{bankroll}")
    print(f"MIN EV threshold:\n{MIN_EV_THRESHOLD}")
    print(f"MIN Position threshold:\n{MAX_POSITION}")

MIN_EV_THRESHOLD = 0.01   # require 1% edge, default = 0
# bankroll = api.get_balance()
bankroll = 100
MAX_POSITION = 0.05

positions = []

for _, trade in opportunities_df.iterrows():

    if trade["best_ev"] < MIN_EV_THRESHOLD:
        print(f"Current trade is less than required ev ({MIN_EV_THRESHOLD}), skipping")
        continue

    # usually cannot short, this only happens when im closing positions
    if trade["best_action"] == "buy_yes_ev" or trade["best_action"] == "sell_no_ev":
        best_action = "buy_yes_ev"
        ev = trade["buy_yes_ev"]
        token_id = trade["yes_token"]
        outcome = "YES"
        kelly = trade["buy_yes_kelly"]
        current_ask = trade["yes_ask"]

    elif trade["best_action"] == "buy_no_ev" or trade["best_action"] == "sell_yes_ev":
        best_action = "buy_no_ev"
        ev = trade["buy_no_ev"]
        token_id = trade["no_token"]
        outcome = "NO"
        kelly = trade["buy_no_kelly"]
        current_ask = trade["no_ask"]

    dollars = bankroll * kelly

    dollars = min(dollars, bankroll * MAX_POSITION)

    # current_balance = api.balance(asset_type="conditional", token_id=token_id)

    # current_size = float(current_balance["balance"])
    current_size = 0

    # inventory cap
    dollars = min(dollars, bankroll * MAX_POSITION - current_size * current_ask)

    if dollars <= 0:
        print("no more dollars to allocate for this trade position, skipping\n")
        continue

    size = dollars / current_ask

    normalized_size = api.normalize_size(size)

    unrealized_pnl = normalized_size * ev

    format_buy_signal(trade, outcome, best_action, normalized_size, current_ask,
                      unrealized_pnl, bankroll, MIN_EV_THRESHOLD, MAX_POSITION)

    confirm_order = (input("Place this order? Type YES to confirm: ") == "YES")
    
    if confirm_order == True:

        price_tick = api.get_tick_size(token_id)
        price = api.round_to_tick(current_ask, price_tick)

        try:
            order = api.place_limit_order(
                token_id=token_id,
                side="BUY",
                price=price,
                size=normalized_size,
                order_type="GTC",
            )
    
            print("\nORDER SUBMITTED\n\n")
            print(order)

            order_row = {
                "order_id": order["clob_order_id"],
                "condition_id": trade["conditionId"],
                "token_id": token_id,
                "outcome": outcome,
                "side": "BUY",
                "price": price,
                "requested_size": normalized_size,
                "order_type": "GTC",
                "status": "OPEN",
                "created_at": datetime.now(timezone.utc),
                "cancelled_at": None,
            }
    
            orders_df = pd.concat([orders_df, pd.DataFrame([order_row])], ignore_index=True)
    
            print("\nORDER RECORDED\n\n")

        except Exception as e:
            print("\nORDER ERROR\n\n")
            print(e)
            break

    else:
        print("\nSKIPPING ORDER\n\n")

no more dollars to allocate for this trade position, skipping

BUY SIGNAL
Market:
Will Bitcoin reach $70,000 by December 31, 2026?


Outcome:
NO


Best Action:
buy_no_ev


Recommended Size:
2.507418


Current Ask:
0.3


Buy Yes EV:
0.014413000000000002


Sell No EV:
0.014413


Buy No EV:
0.0147


Sell Yes EV:
0.014700000000000003


Unrealized Pnl:
0.036859044599999995


Bankroll:
100
MIN EV threshold:
0.01
MIN Position threshold:
0.05
{ TEST } - GET TICK SIZE
{ TEST } - PLACED LIMIT ORDER

ORDER SUBMITTED


{'clob_order_id': 'a031d9cf-5c9c-439a-a2a3-5ed2cb571220', 'replayed': False}

ORDER RECORDED


Current trade is less than required ev (0.01), skipping
Current trade is less than required ev (0.01), skipping
Current trade is less than required ev (0.01), skipping
Current trade is less than required ev (0.01), skipping
Current trade is less than required ev (0.01), skipping
Current trade is less than required ev (0.01), skipping


In [ ]:
def sync_orders(api, orders_df):

    api_orders = api.list_orders()

    for order in api_orders:

        order_id = order["id"]

        mask = orders_df["order_id"] == order_id

        if not mask.any():

            order_row = {
                "order_id": order_id,
                "condition_id": order.get("market"),
                "token_id": order.get("asset_id"),
                "outcome": None,
                "side": order.get("side"),
                "price": order.get("price"),
                "requested_size": order.get("original_size"),
                "filled_size": order.get("size_matched", 0),
                "remaining_size": (order.get("original_size", 0) - order.get("size_matched", 0)),
                "status": order.get("status"),
                "created_at": order.get("created_at"),
                "cancelled_at": None,
            }

            orders_df = pd.concat([orders_df, pd.DataFrame([order_row])], ignore_index=True)

        else:
            idx = orders_df.index[mask][0]

            orders_df.loc[idx, "status"] = order.get("status")
            orders_df.loc[idx, "filled_size"] = order.get("size_matched", 0)
            orders_df.loc[idx, "remaining_size"] = (order.get("original_size", 0) - order.get("size_matched", 0))

    return orders_df

In [ ]:
def get_all_trades(self):
    all_trades = []
    next_cursor = None

    while True:

        response = self.trades(
            next_cursor=next_cursor
        )

        all_trades.extend(response["trades"])

        next_cursor = response.get("next_cursor")

        if not next_cursor:
            break

    return pd.DataFrame(all_trades)

# print(trades.iloc[0])
# {
#     "id": "trade_123",
#     "market": "0xabc...",
#     "asset_id": "12345",
#     "side": "BUY",
#     "price": 0.42,
#     "size": 10,
#     "timestamp": 1720000000,
#     ...
# }

def sync_fills(api, fills_df):

    new_fills = []

    trades = api.get_all_trades()

    for trade in trades:

        fill_id = trade["id"]

        if fill_id in fills_df["fill_id"].values:
            continue

        new_fills.append({
            "fill_id": fill_id,
            "order_id": trade.get("order_id"),
            "condition_id": trade["market"],
            "token_id": trade["asset_id"],
            "outcome": trade.get("outcome"),
            "side": trade["side"],
            "price": float(trade["price"]),
            "shares": float(trade["size"]),
            "fee": float(trade.get("fee", 0)),
            "timestamp": trade["timestamp"],
        })

    # Nothing new
    if not new_fills:
        return fills_df

    # Append new rows
    fills_df = pd.concat([fills_df, pd.DataFrame(new_fills)], ignore_index=True)

    return fills_df

In [ ]:
print(api.health())

In [ ]:
#FIFO matching

from collections import deque

def reconstruct_positions_fifo(fills_df):
    """
    Reconstruct current Polymarket positions using FIFO.

    Expected fills_df columns:
        trade_id
        order_id
        condition_id
        token_id
        outcome
        side
        price
        shares
        fee
        timestamp

    Returns:
        positions_df:
            Current open positions with FIFO cost basis.

        realized_df:
            Realized P&L from SELL trades.
    """

    positions = []
    realized = []

    # Process each token independently.
    for token_id, trades in fills_df.groupby("token_id"):

        # FIFO queue of open BUY lots.
        # Each lot contains:
        #   remaining shares
        #   price
        #   fee_per_share
        buy_lots = deque()

        realized_pnl = 0.0
        realized_shares = 0.0
        realized_fees = 0.0

        # Very important: process chronologically.
        trades = trades.sort_values("timestamp")

        for _, trade in trades.iterrows():

            side = trade["side"]
            shares = float(trade["shares"])
            price = float(trade["price"])
            fee = float(trade.get("fee", 0) or 0)

            if side == "BUY":

                # Store this BUY as a FIFO lot.
                fee_per_share = fee / shares if shares > 0 else 0

                buy_lots.append({
                    "shares": shares,
                    "price": price,
                    "fee_per_share": fee_per_share,
                })

            elif side == "SELL":

                remaining_to_sell = shares
                sell_fee_per_share = fee / shares if shares > 0 else 0

                while remaining_to_sell > 0:

                    if not buy_lots:
                        print(f"SELL exceeds available position for token_id={token_id}")
                        raise ValueError(f"SELL exceeds available position for token_id={token_id}")

                    lot = buy_lots[0]

                    matched_shares = min(remaining_to_sell, lot["shares"])

                    # Cost of the shares being sold.
                    buy_cost = (matched_shares * lot["price"])

                    # Allocate the original BUY fee
                    # to the shares being sold.
                    buy_fee = (matched_shares * lot["fee_per_share"])

                    # Proceeds from the SELL.
                    sell_proceeds = (matched_shares * price)

                    # Allocate SELL fee to this FIFO match.
                    sell_fee = (matched_shares * sell_fee_per_share)

                    # Realized P&L.
                    pnl = (sell_proceeds - sell_fee - buy_cost - buy_fee)

                    realized_pnl += pnl
                    realized_shares += matched_shares
                    realized_fees += buy_fee + sell_fee

                    # Reduce the FIFO lot.
                    lot["shares"] -= matched_shares
                    remaining_to_sell -= matched_shares

                    # Remove empty lot.
                    if lot["shares"] <= 1e-12:
                        buy_lots.popleft()

        # --------------------------------------------------
        # Remaining BUY lots = current position
        # --------------------------------------------------
        remaining_shares = sum(lot["shares"] for lot in buy_lots)

        if remaining_shares <= 0:
            continue

        remaining_cost = sum(lot["shares"] * (lot["price"] + lot["fee_per_share"]) for lot in buy_lots)

        avg_entry_price = (remaining_cost / remaining_shares)

        # Use the first trade for metadata.
        first_trade = trades.iloc[0]

        positions.append({
            "condition_id": first_trade["condition_id"],
            "token_id": token_id,
            "outcome": first_trade.get("outcome"),
            "shares": remaining_shares,
            "cost_basis": remaining_cost,
            "avg_entry_price": avg_entry_price,
            "realized_pnl": realized_pnl,
            "realized_shares": realized_shares,
            "realized_fees": realized_fees,
        })

        realized.append({
            "condition_id": first_trade["condition_id"],
            "token_id": token_id,
            "outcome": first_trade.get("outcome"),
            "realized_shares": realized_shares,
            "realized_pnl": realized_pnl,
            "realized_fees": realized_fees,
        })

    positions_df = pd.DataFrame(positions)
    realized_df = pd.DataFrame(realized)

    return positions_df, realized_df

In [ ]:
def mark_positions_to_market(positions_df, markets_df):
    """
    Add current market prices and unrealized PnL to open positions.

    positions_df columns:
        token_id
        shares
        avg_entry_price

    markets_df columns:
        token_id
        price

    Returns:
        positions_df with:
            current_price
            market_value
            cost_basis
            unrealized_pnl
            unrealized_return
    """

    positions_df = positions_df.copy()

    # Keep only the columns we need from markets
    prices = markets_df[["token_id", "price"]].copy()

    # Make sure there is only one price per token
    prices = prices.drop_duplicates("token_id")

    # Match current market price onto each position
    positions_df = positions_df.merge(
        prices,
        on="token_id",
        how="left",
        suffixes=("", "_market")
    )

    # Rename for clarity
    positions_df = positions_df.rename(
        columns={"price": "current_price"}
    )

    # Current value of position
    positions_df["market_value"] = (
        positions_df["shares"] *
        positions_df["current_price"]
    )

    # Original cost of remaining position
    positions_df["cost_basis"] = (
        positions_df["shares"] *
        positions_df["avg_entry_price"]
    )

    # Unrealized profit/loss
    positions_df["unrealized_pnl"] = (
        positions_df["market_value"] -
        positions_df["cost_basis"]
    )

    # Percentage return
    positions_df["unrealized_return"] = (
        positions_df["unrealized_pnl"] /
        positions_df["cost_basis"]
    )

    return positions_df

In [ ]:
def calculate_equity(
    positions_df,
    realized_df,
    equity_df
):
    """
    Calculate the current account equity and P&L.

    Uses the last row of equity_df to determine
    previous_equity and previous_timestamp.
    """

    # ---------------------------------------------------------
    # 0. Get previous equity snapshot
    # ---------------------------------------------------------

    if equity_df.empty:
        previous_equity = None
    else:
        previous_row = equity_df.iloc[-1]
        previous_equity = float(previous_row["equity"])

    # ---------------------------------------------------------
    # 1. Get current cash
    # ---------------------------------------------------------
    balance = api.sync_balance()
    cash = float(balance["balance"])

    # ---------------------------------------------------------
    # 2. Market value of open positions
    # ---------------------------------------------------------
    if positions_df.empty:
        market_value = 0.0
        unrealized_pnl = 0.0

    else:
        if "market_value" in positions_df.columns:
            market_value = positions_df["market_value"].sum()
        else:
            market_value = (positions_df["shares"] * positions_df["current_price"]).sum()

        if "unrealized_pnl" in positions_df.columns:
            unrealized_pnl = positions_df["unrealized_pnl"].sum()

        else:
            unrealized_pnl = (positions_df["shares"] * (
                    positions_df["current_price"]
                    - positions_df["avg_entry_price"]
                )
            ).sum()

    # ---------------------------------------------------------
    # 3. Realized P&L
    # ---------------------------------------------------------
    if realized_df.empty:
        realized_pnl = 0.0
    else:
        realized_pnl = realized_df["realized_pnl"].sum()

    # ---------------------------------------------------------
    # 4. Total equity
    # ---------------------------------------------------------
    equity = cash + market_value

    # ---------------------------------------------------------
    # 5. Return
    # ---------------------------------------------------------
    if previous_equity is None or previous_equity == 0:
        daily_return = None
    else:
        daily_return = equity / previous_equity - 1

    # ---------------------------------------------------------
    # 6. Return new row
    # ---------------------------------------------------------
    equity_row = {
        "timestamp": datetime.now(timezone.utc),
        "cash": cash,
        "market_value": market_value,
        "equity": equity,
        "realized_pnl": realized_pnl,
        "unrealized_pnl": unrealized_pnl,
        "daily_return": daily_return,
    }

    equity_df = pd.concat([equity_df, pd.DataFrame([equity_row])], ignore_index=True)

    return equity_df

In [ ]:
while True:

     # 1. Get newly executed trades
    fills_df = sync_fills(api, fills_df)

    # 2. Reconstruct portfolio
    # 3. Calculate realized P&L
    positions_df, realized_df = reconstruct_positions_fifo(fills_df)

    # 4. Get latest order state
    orders_df = sync_orders(api, orders_df)

    # 5. Mark positions to market
    positions_df = mark_positions_to_market(positions_df, markets_df)

    # 6. Calculate equity
    equity_df = calculate_equity(positions_df, realized_df, equity_df)

    # 7. Risk management
    run_risk_management(
        positions_df,
        markets_df,
    )

    time.sleep(300)

In [ ]:
def save(df):

    date = datetime.now().strftime("%Y%m%d")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    DATA_DIR = f"data/{date}"
    os.makedirs(DATA_DIR, exist_ok=True)
    filename = f"{DATA_DIR}/markets_{timestamp}.parquet"

    df.to_parquet(filename, engine="fastparquet", index=False)

    print('df saved at: ', filename)

save(markets_df)

markets_df, opportunities_df

In [ ]:
df = []

df.append({
        "timestamp": 1.0,
        "cash": 1.0,
        "market_value": 1.0,
        "equity": 1.0,
        "realized_pnl": 1.0,
        "unrealized_pnl": 1.0,
        "daily_return": 1.0,
    })

df = pd.DataFrame(df)

DATA_DIR = f"data"
os.makedirs(DATA_DIR, exist_ok=True)
filename = f"{DATA_DIR}/equity.parquet"

df.to_parquet(filename, engine="fastparquet", index=False)

print('df saved at: ', filename)

In [ ]:
#                  Polymarket    Model (exchanges: deribit, bybit, okx)    Edge

# 80k Dec-26        22%          20%       +2%
# 90k Dec-26        14%          12%       +2%
# 100k Dec-26        9.5%         5.9%     +3.6%
# 110k Dec-26        7%           4%       +3%
# 120k Dec-26        5%           2%       +3%

In [ ]:
while True:

    # 1. Manage inventory
    positions = api.get_positions()

    for pos in positions:

        market = api.get_market(pos.condition_id)

        if pos.outcome == "Yes":
            hold_ev = market.buy_yes_ev
            exit_ev = market.sell_yes_ev

        else:
            hold_ev = market.buy_no_ev
            exit_ev = market.sell_no_ev

        if hold_ev < MIN_EXIT_THRESHOLD:
            api.close_position(pos)


    # 2. Find new opportunities
    opportunities = scanner.scan_market()

    for trade in opportunities:

        if trade.best_ev > MIN_EV:
            api.place_limit_order(...)


In [ ]:
# Keep trading journal

# Your thesis.
# Why you think the market is mispriced.
# Position size.
# Exit criteria.
# What actually happened.

In [ ]:
# Stage 2 — Semi-automated execution

# The bot does:

# find opportunities
# calculate size
# prepare orders

# You approve:

# BUY YES
# Market: BTC above $150k
# Price: 0.43
# Size: $500
# Expected edge: +8%

# Click "confirm".

# This is useful because prediction markets can have:

# ambiguous wording
# resolution risks
# sudden news events

In [ ]:
# Day 1 — API connection + market ingestion

# Goal:

# Can I pull markets automatically?

# Build:

# get_markets()

# Output:

# {
#  "condition_id": "...",
#  "question": "Will X happen?",
#  "tokens": [
#     {
#       "token_id": "YES",
#       "price": 0.42
#     },
#     {
#       "token_id": "NO",
#       "price": 0.58
#     }
#  ]
# }

# Store:

# markets

# condition_id
# question
# yes_token
# no_token
# created_time
# Day 2 — Historical snapshots

# Goal:

# Can I reconstruct what the market looked like yesterday?

# Every 5 minutes:

# while True:

#     markets = api.get_markets()

#     for m in markets:
#         database.save_snapshot(m)

#     sleep(300)

# Database:

# market_snapshots

# timestamp
# condition_id
# yes_price
# no_price
# volume
# Day 3 — Order book collection

# Now collect microstructure data.

# For selected markets:

# get_orderbook(token_id)

# Store:

# orderbook_snapshots

# timestamp

# token_id

# best_bid
# best_ask

# bid_depth
# ask_depth

# Calculate:

# Spread
# spread=ask−bid

# Example:

# Bid:
# 0.42

# Ask:
# 0.46

# Spread:
# 4 cents
# Day 4 — Build your scanner

# Your first scanner should be dumb but useful.

# Signal 1: Large moves
# if abs(price_change_24h) > 0.10:
#     flag()

# Example:

# AI model release

# Yesterday:
# 35%

# Today:
# 52%

# Move:
# +17%
# Signal 2: Liquidity opportunities
# if spread > 0.08:
#     flag()
# Signal 3: Volume spikes
# if volume_today > 5 * average_volume:
#     flag()

# Your output:

# TOP MARKETS TO REVIEW

# 1.
# Question:
# Will Fed cut rates?

# Price:
# 42%

# 24h move:
# +12%

# Reason:
# Large movement


# 2.
# Question:
# Will Company X acquire Y?

# Price:
# 33%

# Spread:
# 11 cents

# Reason:
# Wide market
# Day 5 — Add your probability workflow

# Do NOT automate this yet.

# Create a manual table:

# trade_journal.csv

# market,current_price,my_probability,edge,reason
# Fed cut,0.42,0.55,0.13,"Inflation falling"
# AI launch,0.35,0.45,0.10,"Company comments"

# The key question:

# Market probability:
# 42%

# My probability:
# 55%

# Difference:
# +13%
# Day 6 — Paper trading

# Before your C++ engine touches anything:

# Create:

# paper_buy(
#     market,
#     price,
#     size
# )

# Track:

# Position:
# YES Fed cut

# Entry:
# 42c

# Size:
# $100

# Current:
# 48c

# P&L:
# +$14
# Day 7 — Analytics

# Calculate:

# Return
# profit / capital
# Drawdown

# Largest loss from peak.

# Sortino

# Track:

# returns
# negative returns only

# Your output:

# Paper Portfolio

# Trades:
# 18

# Win rate:
# 61%

# Return:
# +8.4%

# Max drawdown:
# -2.1%

# Sortino:
# 2.4

In [ ]:
{'id': '701496', 'question': 'Will Bitcoin reach $100,000 by December 31, 2026?', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 'slug': 'will-bitcoin-reach-100000-by-december-31-2026-571-361-361', 
 'resolutionSource': '', 'endDate': '2027-01-01T05:00:00Z', 'liquidity': '94083.1351', 'startDate': '2025-11-24T19:07:17.691Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
'description': 'This market will immediately resolve to "Yes" if any Binance 1 minute candle for Bitcoin (BTC/USDT) between November 24, 2025, 14:00 and December 31, 2026, 23:59 in the ET timezone has a final "High" price equal to or greater than the price specified in the title. Otherwise, this market will resolve to "No."\n\nThe resolution source for this market is Binance, specifically the BTC/USDT "High" prices available at https://www.binance.com/en/trade/BTC_USDT, with the chart settings on "1m" for one-minute candles selected on the top bar.\n\nPlease note that the outcome of this market depends solely on the price data from the Binance BTC/USDT trading pair. Prices from other exchanges, different trading pairs, or spot markets will not be considered for the resolution of this market.', 
'outcomes': '["Yes", "No"]', 'outcomePrices': '["0.095", "0.905"]', 'volume': '2370982.5583320004', 'active': True, 'closed': False, 'marketMakerAddress': '', 'createdAt': '2025-11-24T18:55:12.725029Z', 'updatedAt': '2026-08-02T10:54:52.526927Z', 
'new': False, 'featured': False, 'submitted_by': '0x91430CaD2d3975766499717fA0D66A78D814E5c5', 'archived': False, 'resolvedBy': '0x65070BE91477460D8A7AeEb94ef92fe056C2f2A7', 'restricted': True, 'groupItemTitle': '↑ 100,000', 'groupItemThreshold': '13', 
'questionID': '0x3c9be67d4b90291760ac3bffc1f9470a1966e5c1f3e99131333170e3469bd023', 'enableOrderBook': True, 'orderPriceMinTickSize': 0.01, 'orderMinSize': 5, 'volumeNum': 2370982.5583320004, 'liquidityNum': 94083.1351, 'endDateIso': '2027-01-01', 
'startDateIso': '2025-11-24', 'hasReviewedDates': True, 'volume24hr': 1365.610655, 'volume1wk': 66640.791618, 'volume1mo': 228733.97684700004, 'volume1yr': 2370982.5583320004, 
'clobTokenIds': '["56078938060096976448086754249497300447360333783952000147427828224794011030104", "11291662904897713174667903388388696640643610556195928998276904135282270136756"]', 
'comboStatus': 'disabled', 'umaBond': '500', 'umaReward': '5', 'volume24hrClob': 1365.610655, 'volume1wkClob': 66640.791618, 'volume1moClob': 228733.97684700004, 'volume1yrClob': 2370982.5583320004, 'volumeClob': 2370982.5583320004, 
'liquidityClob': 94083.1351, 'makerBaseFee': 1000, 'takerBaseFee': 1000, 'customLiveness': 0, 'acceptingOrders': True, 'negRisk': False, 'negRiskRequestID': '', 
'events': [{'id': '89502', 'ticker': 'what-price-will-bitcoin-hit-before-2027', 'slug': 'what-price-will-bitcoin-hit-before-2027', 
            'title': 'What price will Bitcoin hit in 2026?', 'description': 'What price will Bitcoin hit before 2027?  ', 
            'resolutionSource': '', 'startDate': '2025-11-24T19:07:12.848Z', 'creationDate': '2025-11-24T19:13:13.705687Z', 'endDate': '2027-01-01T05:00:00Z', 
            'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
            'active': True, 'closed': False, 'archived': False, 'new': False, 'featured': False, 'restricted': True, 'liquidity': 2695251.24076, 'volume': 50935482.262915, 
            'openInterest': 9503925.568983998, 'createdAt': '2025-11-24T18:55:05.597959Z', 'updatedAt': '2026-08-02T10:55:09.654318Z', 'competitive': 0.9999750006249843, 
            'volume24hr': 130499.32620200001, 'volume1wk': 2062901.9714630004, 'volume1mo': 7099616.726362999, 'volume1yr': 49102712.92022599, 'enableOrderBook': True, 
            'liquidityClob': 2695251.24076, 'negRisk': False, 'commentCount': 0, 'series': [{'id': '10016', 'ticker': 'bitcoin-hit-price-monthly', 'slug': 'bitcoin-hit-price-monthly', 
                                                                                        'title': 'Bitcoin Hit Price Monthly', 'seriesType': 'single', 'recurrence': 'monthly', 
                                                                                        'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'active': True, 'closed': False, 'archived': False, 'featured': False, 'restricted': True, 
                                                                                        'createdAt': '2025-01-31T22:03:50.00441Z', 'updatedAt': '2026-08-02T10:55:28.185077Z', 
                                                                                        'volume24hr': 601000.637578, 'volume': 51492794.133888, 'liquidity': 3438609.2653, 'commentCount': 6318, 
                                                                                        'requiresTranslation': False}], 
            'cyom': False, 'showAllOutcomes': True, 'showMarketImages': False, 'enableNegRisk': False, 'automaticallyActive': True, 'seriesSlug': 'bitcoin-hit-price-monthly', 
            'gmpChartMode': 'default', 'negRiskAugmented': False, 'estimateValue': True, 'cantEstimate': True, 'cumulativeMarkets': False, 'pendingDeployment': False, 'deploying': False, 
            'requiresTranslation': False, 'eventMetadata': {'context_requires_regen': True}, 'version': 'v1'}], 

'ready': False, 'funded': False, 'acceptingOrdersTimestamp': '2025-11-24T19:06:55Z', 
'cyom': False, 'competitive': 0.8590880780051975, 'pagerDutyNotificationEnabled': False, 'approved': True, 'clobRewards': [{'id': '418394', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 
                                                                                                                        'assetAddress': '0xc011a7e12a19f7b1f670d46f03b03f3342e82dfb', 'rewardsAmount': 0, 'rewardsDailyRate': 0.001, 
                                                                                                                        'startDate': '2026-06-03', 'endDate': '2500-12-31'}], 
'rewardsMinSize': 0, 'rewardsMaxSpread': 0, 'spread': 0.01, 'oneMonthPriceChange': -0.01, 'lastTradePrice': 0.09, 'bestBid': 0.09, 'bestAsk': 0.1, 'automaticallyActive': True, 
'clearBookOnStart': True, 'seriesColor': '', 'showGmpSeries': False, 'showGmpOutcome': False, 'manualActivation': False, 'negRiskOther': False, 'umaResolutionStatuses': '[]', 
'pendingDeployment': False, 'deploying': False, 'deployingTimestamp': '2025-11-24T19:06:23.727362Z', 'rfqEnabled': False, 'holdingRewardsEnabled': True, 'feesEnabled': True, 
'requiresTranslation': False, 'feeType': 'crypto_fees_v2', 'feeSchedule': {'exponent': 1, 'rate': 0.07, 'takerOnly': True, 'rebateRate': 0.2}, 'version': 'v1'}
["0.095", "0.905"]